In [ ]:
# Performance config
import os

CPU_THREADS = min(32, os.cpu_count() or 32)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "true"

REQUIRE_CUDA = True  # set False for CPU-only notebooks

print(f"CPU threads set to: {CPU_THREADS}")

try:
    import torch
except Exception as e:
    torch = None
    if REQUIRE_CUDA:
        raise RuntimeError("CUDA required but torch is not available.") from e

if torch is not None:
    torch.set_num_threads(CPU_THREADS)
    torch.set_num_interop_threads(min(4, CPU_THREADS))
    if REQUIRE_CUDA and not torch.cuda.is_available():
        raise RuntimeError("CUDA required but not available.")
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        print("CUDA device:", torch.cuda.get_device_name(0))
    else:
        print("CUDA not available; running on CPU.")


# Kvasir-VQA x1 — CLIP embeddings + linear classifier

Compute frozen CLIP image embeddings for the Kvasir-VQA x1 splits and train a shallow classifier on top-K answers. This mirrors the HyperKvasir CLIP baseline, adapted to the VQA metadata.

In [1]:
from pathlib import Path
import json
import random
from typing import Tuple

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report

from transformers import CLIPProcessor, CLIPModel

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


2026-01-18 02:54:22.545480: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-18 02:54:22.545513: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-18 02:54:22.546610: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-18 02:54:22.553449: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-18 02:54:23.370800: W tensorflow/compiler/tf2

In [2]:
# Paths & config

def find_kvasir_x1_root() -> Path:
    import os
    env_root = os.environ.get("KVASIR_VQA_X1_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / "0_dataset_prep").exists():
            return p
        raise RuntimeError(f"KVASIR_VQA_X1_ROOT set but missing 0_dataset_prep: {p}")

    if "__file__" in globals():
        p = Path(__file__).resolve()
        root = p.parents[2]
        if root.name == "Kvasir_VQA_x1" and (root / "0_dataset_prep").exists():
            return root

    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if p.name == "Kvasir_VQA_x1" and (p / "0_dataset_prep").exists():
            return p

    raise RuntimeError(
        "Could not locate Kvasir_VQA_x1 dataset root. \n"
        "Run this notebook from within the Kvasir_VQA_x1 folder, \n"
        "or set KVASIR_VQA_X1_ROOT."
    )

DATA_ROOT = find_kvasir_x1_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"
OUT_DIR = DATA_ROOT / "2_modeling" / "07_clip_linear" / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "openai/clip-vit-base-patch32"  # swap to medical CLIP if available
BATCH_SIZE = 8
NUM_WORKERS = 0  # bump if you want multiprocessing
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MAX_TRAIN_SAMPLES = None
MAX_VAL_SAMPLES = None
MAX_TEST_SAMPLES = None
TOP_K_ANSWERS = 20  # restrict classification to most common answers

print("Device:", DEVICE)
print("Data root:", DATA_ROOT)
print("Output dir:", OUT_DIR)


Device: cuda
Data root: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1
Output dir: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/07_clip_linear/out


In [3]:
# Load metadata
meta = pd.read_csv(META_CSV)
images_base = DATA_ROOT / "0_dataset_prep"
meta["image_path"] = meta["image_path"].apply(
    lambda p: str((images_base / p).resolve()) if not Path(p).is_absolute() else p
)

if "split" not in meta.columns:
    raise RuntimeError("Missing 'split' column. Run dataset prep split step first.")

# Drop missing answers and missing images
meta = meta.dropna(subset=["answer"]).reset_index(drop=True)
meta = meta[meta["image_path"].apply(lambda p: Path(p).exists())].reset_index(drop=True)

# Restrict to top-K answers for classification
answer_counts = meta["answer"].value_counts()
top_answers = set(answer_counts.head(TOP_K_ANSWERS).index)
meta = meta[meta["answer"].isin(top_answers)].reset_index(drop=True)

answer_to_id = {ans: i for i, ans in enumerate(sorted(top_answers))}
id_to_answer = {v: k for k, v in answer_to_id.items()}
meta["label_id"] = meta["answer"].map(answer_to_id)

train_df = meta[meta["split"] == "train"].reset_index(drop=True)
val_df   = meta[meta["split"] == "validation"].reset_index(drop=True)
test_df  = meta[meta["split"] == "test"].reset_index(drop=True)

if MAX_TRAIN_SAMPLES:
    train_df = train_df.sample(min(MAX_TRAIN_SAMPLES, len(train_df)), random_state=SEED).reset_index(drop=True)
if MAX_VAL_SAMPLES:
    val_df = val_df.sample(min(MAX_VAL_SAMPLES, len(val_df)), random_state=SEED).reset_index(drop=True)
if MAX_TEST_SAMPLES:
    test_df = test_df.sample(min(MAX_TEST_SAMPLES, len(test_df)), random_state=SEED).reset_index(drop=True)

print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})


{'train': 41789, 'val': 5317, 'test': 5294}


In [4]:
# Dataset & dataloader
class ImageDS(Dataset):
    def __init__(self, df: pd.DataFrame, processor):
        self.paths = df["image_path"].tolist()
        self.labels = df["label_id"].astype(int).tolist()
        self.processor = processor
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        encoded = self.processor(images=img, return_tensors="pt")
        return encoded, self.labels[idx]

def collate_fn(batch):
    pixel_values = torch.cat([b[0]["pixel_values"] for b in batch], dim=0)
    labels = torch.tensor([b[1] for b in batch], dtype=torch.long)
    return pixel_values, labels


In [5]:
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
try:
    clip_model = CLIPModel.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if torch.cuda.is_available() else None,
    )
    clip_model.to(DEVICE)
except RuntimeError as e:
    print("GPU load failed (likely OOM). Falling back to CPU. Error:", e)
    clip_model = CLIPModel.from_pretrained(MODEL_NAME)
    clip_model.to(torch.device("cpu"))
    # Also shrink batch to limit CPU RAM
    global BATCH_SIZE
    BATCH_SIZE = min(BATCH_SIZE, 4)

clip_model.eval()
print("Loaded CLIP on", next(clip_model.parameters()).device, "batch_size=", BATCH_SIZE)


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loaded CLIP on cuda:0 batch_size= 8


In [6]:
# Embedding helper
def extract_split(df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    ds = ImageDS(df, processor)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn)
    feats, labels = [], []
    with torch.no_grad():
        for pixels, y in tqdm(dl, desc="embed", leave=False):
            pixels = pixels.to(DEVICE)
            out = clip_model.get_image_features(pixel_values=pixels)
            feats.append(out.cpu().numpy())
            labels.append(y.numpy())
    return np.concatenate(feats, axis=0), np.concatenate(labels, axis=0)


In [7]:
# Extract embeddings per split
print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})

train_X, train_y = extract_split(train_df)
val_X,   val_y   = extract_split(val_df)
test_X,  test_y  = extract_split(test_df)

np.savez(OUT_DIR / "clip_embeddings.npz", train_X=train_X, val_X=val_X, test_X=test_X, train_y=train_y, val_y=val_y, test_y=test_y)
print("Saved embeddings to", OUT_DIR / "clip_embeddings.npz")


{'train': 41789, 'val': 5317, 'test': 5294}


embed:   0%|          | 0/5224 [00:00<?, ?it/s]

embed:   0%|          | 0/665 [00:00<?, ?it/s]

embed:   0%|          | 0/662 [00:00<?, ?it/s]

Saved embeddings to /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/07_clip_linear/out/clip_embeddings.npz


In [8]:
# Train logistic regression on frozen features
scaler = StandardScaler()
train_X_scaled = scaler.fit_transform(train_X)
val_X_scaled = scaler.transform(val_X)
test_X_scaled = scaler.transform(test_X)

clf = LogisticRegression(max_iter=1000, n_jobs=-1, multi_class="multinomial")
clf.fit(train_X_scaled, train_y)

val_preds = clf.predict(val_X_scaled)
test_preds = clf.predict(test_X_scaled)

val_acc = accuracy_score(val_y, val_preds)
test_acc = accuracy_score(test_y, test_preds)
val_f1 = f1_score(val_y, val_preds, average="macro")
test_f1 = f1_score(test_y, test_preds, average="macro")

metrics = {
    "val": {"accuracy": val_acc, "macro_f1": val_f1},
    "test": {"accuracy": test_acc, "macro_f1": test_f1},
}

with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(metrics)


/home/aristotle/anaconda3/envs/vqa-rag/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1264: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


{'val': {'accuracy': 0.274967086703028, 'macro_f1': 0.08867276109641865}, 'test': {'accuracy': 0.28447298828862866, 'macro_f1': 0.08199591298362865}}


In [9]:
# Per-class report
labels = sorted(answer_to_id.values())
report = classification_report(
    test_y,
    test_preds,
    labels=labels,
    target_names=[id_to_answer[i] for i in labels],
    output_dict=True,
    zero_division=0,
)
with open(OUT_DIR / "classification_report.json", "w") as f:
    json.dump(report, f, indent=2)
print(pd.DataFrame(report).T.head())


                                                    precision    recall  \
0                                                    0.000000  0.000000   
1                                                    0.176829  0.065169   
11-20mm                                              0.034483  0.043478   
2                                                    0.166667  0.009009   
center; center-left; center-right; lower-center...   0.000000  0.000000   

                                                    f1-score  support  
0                                                   0.000000    456.0  
1                                                   0.095238    445.0  
11-20mm                                             0.038462     46.0  
2                                                   0.017094    111.0  
center; center-left; center-right; lower-center...  0.000000    180.0  
